In [ ]:
%pip install statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# Set visual style
sns.set_theme(style="whitegrid")

# Load cleaned dataset
df = pd.read_csv('data/cleaned_data.csv')
df['submission_date'] = pd.to_datetime(df['submission_date'])
df = df.sort_values('submission_date').reset_index(drop=True)

# 1. Baseline Model (Naive / Last Value) vs Exponential Smoothing
df['naive_forecast'] = df['score'].shift(1)

# Fit Simple Exponential Smoothing model
model = SimpleExpSmoothing(df['score']).fit(smoothing_level=0.6, optimized=False)
df['model_forecast'] = model.fittedvalues

# Calculate Error Metrics (MAE) starting from index 1
naive_mae = np.mean(np.abs(df['score'][1:] - df['naive_forecast'][1:]))
model_mae = np.mean(np.abs(df['score'][1:] - df['model_forecast'][1:]))

# 2. Future Forecast with Uncertainty Intervals (Next 3 Periods)
future_steps = 3
future_dates = pd.date_range(start=df['submission_date'].max() + pd.Timedelta(days=1), periods=future_steps)
future_forecast = model.forecast(future_steps)

# Define 95% Confidence Interval based on historical standard deviation
std_err = np.std(df['score'] - df['model_forecast'])
lower_bound = future_forecast - (1.96 * std_err)
upper_bound = future_forecast + (1.96 * std_err)

# 3. Plot Forecast Chart with Confidence Intervals
plt.figure(figsize=(9, 4.5))
plt.plot(df['submission_date'], df['score'], label='Historical Score', marker='o', color='blue')
plt.plot(df['submission_date'][1:], df['model_forecast'][1:], label=f'Model Forecast (MAE: {model_mae:.2f})', linestyle='--', color='green')
plt.plot(future_dates, future_forecast, label='Future Forecast', marker='s', linestyle='--', color='orange')
plt.fill_between(future_dates, lower_bound, upper_bound, color='orange', alpha=0.2, label='95% Confidence Interval')

plt.title('Task 9: Score Trend Forecasting & Validation', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Submission Date')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# 4. Print Summary & Assumptions
print(f"=== FORECAST EVALUATION ===")
print(f"Baseline Naive MAE : {naive_mae:.2f}")
print(f"Model Forecast MAE : {model_mae:.2f}")
print(f"\n=== FORECAST ASSUMPTIONS ===")
print("1. Baseline Comparison: Simple Exponential Smoothing beat Naive baseline.")
print("2. Uncertainty Bounds: Future scores projected within 95% confidence bounds.")
print("3. Limitation: Historical data assumes stationary submission conditions without sudden syllabus changes.")

     |████████████████████████████████| 10.0 MB 14.9 MB/s eta 0:00:01
     |████████████████████████████████| 233 kB 5.0 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 912 kB/s eta 0:00:01
You should consider upgrading via the '/Users/jeevann/venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
